# Production Monitoring: Automated Quality at Scale

MLflow's production monitoring automatically runs quality assessments on a sample of your production traffic, ensuring your GenAI app maintains high quality standards without manual intervention. MLflow lets you use the **same** metrics you defined for offline evaluation in production, enabling you to have consistent quality evaluation across your entire application lifecycle - dev to prod.

Key benefits:

- **Automated evaluation** - Run LLM judges on production traces with configurable sampling rates
- **Continuous quality assessment** - Monitor quality metrics in real-time without disrupting user experience
- **Cost-effective monitoring** - Smart sampling strategies to balance coverage with computational cost

Production monitoring enables you to deploy confidently, knowing that you will proactively detect issues so you can address them before they cause a major impact to your users.

![monitoring-overview](https://i.imgur.com/wv4p562.gif)

This notebook demonstrates how to take the quality metrics you created in development and turn them into production monitoring with MLflow's scorer registration system for the banking customer support assistant.


## Install packages (only required if running in a Databricks Notebook)

In [ ]:
%pip install -U -r ../requirements.txt
dbutils.library.restartPython()

## Environment Setup

Load environment variables and verify MLflow configuration.


In [ ]:
import sys
sys.path.append('../')
sys.path.append('../../')

import os
import mlflow
from mlflow_demo.utils import *

# Configure for the current participant: derives the object prefix, prompt name,
# model service, and experiment from YOUR username. The widgets only carry the
# shared catalog/schema/warehouse.
setup_databricks_notebook_env()

print('=== Environment Setup ===')
print(f'DATABRICKS_HOST: {os.getenv("DATABRICKS_HOST")}')
print(f'MLFLOW_EXPERIMENT_ID: {os.getenv("MLFLOW_EXPERIMENT_ID")}')
print(f'LLM_MODEL: {os.getenv("LLM_MODEL")}')
print(f'UC_CATALOG: {os.getenv("UC_CATALOG")}')
print(f'UC_SCHEMA: {os.getenv("UC_SCHEMA")}')
print(f'MLFLOW_TRACING_SQL_WAREHOUSE_ID: {os.getenv("MLFLOW_TRACING_SQL_WAREHOUSE_ID")}')
print('Environment variables loaded successfully!')

import logging
logging.getLogger('urllib3').setLevel(logging.ERROR)
logging.getLogger('mlflow').setLevel(logging.ERROR)

In [ ]:
from mlflow_demo.utils import generate_trace_links

# From Offline Evaluation to Online Monitoring

The key insight of MLflow's production monitoring is simple: **the same scorers you use for offline evaluation automatically become your online monitoring**. No need to rebuild, reconfigure, or rewrite your quality assessment logic.

## The "Same Metrics Everywhere" Approach

- **Development**: Use Guidelines, Safety, and custom scorers to evaluate quality
- **Testing**: Run the same scorers on evaluation datasets
- **Production**: Register those exact scorers for automated monitoring
- **Analysis**: Compare quality using consistent metrics across all stages

In this tutorial, we'll take the banking scorers we created in the previous notebook and turn them into production monitoring in just a few lines of code.

**Documentation**

- [Run Scorers in Production](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/run-scorer-in-prod) - Production monitoring setup

Run the next cells to recreate and register your quality metrics for production monitoring!


# Step 1: Register Quality Scorers for Monitoring

Let's register the banking scorers from the evaluation module for production monitoring. This demonstrates how your development metrics seamlessly become production monitoring.

Run the next cells to register your scorers!


## IMPORTANT: Do NOT delete all registered scorers

**WARNING**: Only delete scorers that start with your participant prefix (e.g., `p01_`). Other participants' scorers must not be affected. If SCORER_PREFIX is not set, skip deletion entirely to avoid removing shared scorers in a workshop environment.


In [ ]:
from mlflow.genai.scorers import list_scorers, delete_scorer

prefix = os.environ.get('SCORER_PREFIX', '')
if prefix:
    existing = list_scorers()
    for s in existing:
        if s.name.startswith(f'{prefix}_'):
            print(f'Deleting existing scorer: {s.name}')
            delete_scorer(name=s.name)
    print(f'Cleaned up scorers with prefix: {prefix}_')
else:
    print('No SCORER_PREFIX set. Skipping scorer deletion to avoid removing shared scorers.')

Now, we will register and start the scorers using namespaced names to avoid collisions in the shared workspace.

In [ ]:
from mlflow.genai.scorers import ScorerSamplingConfig
from mlflow.tracing import set_databricks_monitoring_sql_warehouse_id
from mlflow_demo.evaluation.evaluator import get_scorers

prefix = os.environ.get('SCORER_PREFIX', '')
scorers = get_scorers(prefix=prefix)

warehouse_id = os.getenv('MLFLOW_TRACING_SQL_WAREHOUSE_ID')
if warehouse_id:
  set_databricks_monitoring_sql_warehouse_id(sql_warehouse_id=warehouse_id)
  print(f'Monitoring SQL warehouse configured: {warehouse_id}')
else:
  print('MLFLOW_TRACING_SQL_WAREHOUSE_ID not set - monitoring may not function correctly')

print('Registering and starting production monitoring scorers...')

for scorer in scorers:
  scorer.register()
  scorer.start(sampling_config=ScorerSamplingConfig(sample_rate=1))
  print(f'Scorer registered and started: {scorer.name}')

print('All production monitoring scorers are now active!')

# Step 2: View Monitoring Results in MLflow UI

Let's issue a few queries so we can see the monitoring results in the MLflow Trace UI. Since monitoring runs every ~15 minutes, please wait ~20 mins after running these queries to check the UI.


In [ ]:
from mlflow_demo.agent.agent import AGENT
from mlflow.types.responses import Message, ResponsesAgentRequest

banking_questions = [
  'What is my current account balance?',
  'Show me my recent transactions for the last 30 days.',
  'What are the terms of my auto loan?',
]

for question in banking_questions:
  print(f'\nAsking: {question}')
  request = ResponsesAgentRequest(
    input=[Message(role='user', content=question)]
  )
  AGENT.predict(request)
  trace_id = mlflow.get_last_active_trace_id()
  if trace_id:
    generate_trace_links(trace_id);

# Exercise 1

Change the sample_rate from 1.0 to 0.5. What are the trade-offs for a banking application where accuracy is critical?


# Exercise 2

Write a code-based latency scorer that flags responses taking longer than 10 seconds and register it for monitoring.


# Summary and Next Steps

Congratulations! You've successfully implemented production monitoring for the banking customer support assistant using MLflow 3.x.

## What You've Accomplished

- **Connected offline to online** - Used the same scorers from development in production monitoring
- **Registered quality scorers** - Set up namespaced scorers for automated monitoring
- **Generated test traces** - Issued banking queries to trigger monitoring

**Continue Learning**

- [Production Monitoring Guide](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/run-scorer-in-prod) - Complete setup documentation
- [MLflow GenAI Evaluation](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/) - Full evaluation and monitoring workflow
